# Phase 2a — Custom Scoring

nflverse ships `fantasy_points` (standard) and `fantasy_points_ppr` (full PPR).
This league is **0.5 PPR with several non-standard rules**, so neither column is
a valid model target or a valid baseline against Sleeper's projections.

This notebook builds and validates a scorer that reproduces the league's rules
exactly, by diffing against what Sleeper actually awarded in completed weeks.

**Validated result:** 100% exact match on 2025 weeks 5, 8, 10, 12, 15 — every
rostered player, every position.

**Rules discovered by validation** (none of them documented anywhere):

| Rule | What it actually does |
|---|---|
| `fum` | Counts `fumbles_total`, not the sum of rushing/receiving/sack fumbles |
| `fum` + `fum_lost` | Stack — a lost fumble costs −2, a self-recovered one −1 |
| `fum_rec` | Does **not** apply to offensive players (19/19 confirmed) |
| `fgm_yds_over_30` | Per kick, not on aggregate distance |
| `xpmiss` | Blocked PATs count as misses |
| `fgmiss` | Only applies to misses under 50 yards |
| `pass_int_td` | Needs play-by-play; the scoring team must be the defense |


## Setup

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import logging
logging.basicConfig(level=logging.INFO, format='%(message)s')

import pandas as pd
pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 200)

In [ ]:
from src.ingest import (
    get_weekly_stats, get_pbp, get_id_crosswalk, get_sleeper_league,
)
from src.features import (
    compute_custom_score, scoring_coverage_report,
    validate_against_sleeper, add_pick_six_column, kicker_miss_audit,
)

## Configuration

`SEASONS` is the history we model on. `LEAGUE_ID_2025` is used for validation
because we're checking against completed 2025 weeks — the 2026 league has a
different id and no results yet.

In [ ]:
SEASONS        = [2024, 2025]
LEAGUE_ID_2025 = "1250182471429931008"
LEAGUE_ID_2026 = "1389706592789733376"

VALIDATION_WEEKS = [5, 8, 10, 12, 15]
SKILL_POSITIONS  = ['QB', 'RB', 'WR', 'TE']

## 1. Load data

All cache hits — nothing re-downloads.

In [ ]:
weekly    = get_weekly_stats(SEASONS)
pbp       = get_pbp(SEASONS)
crosswalk = get_id_crosswalk()
league    = get_sleeper_league(LEAGUE_ID_2025)
scoring   = league['scoring_settings']

print(f"weekly: {len(weekly):,} rows | pbp: {len(pbp):,} rows")

## 2. Filter to regular season

Weeks 19+ are playoffs. They're real games with real stats, but usage patterns
differ and fantasy seasons end well before them — training on them would teach
the model from games that don't count.

In [ ]:
reg = weekly[weekly['season_type'] == 'REG'].copy()
print(f"{len(reg):,} regular-season rows "
      f"({len(weekly) - len(reg):,} postseason dropped)")

## 3. Add pick-sixes from play-by-play

The one active scoring rule with no weekly-stats column. A pick-six is an
interception returned for a touchdown, charged to the passer — and crucially,
**the scoring team must be the defense**. Without that condition, a defender
fumbling the return into his own end zone counts as a pick-six when it's the
opposite.

In [ ]:
reg = add_pick_six_column(reg, pbp)
print(f"Pick-sixes across {SEASONS}: {int(reg['pass_int_tds'].sum())}")

## 4. Coverage report

Which of the league's active rules can we actually compute? Read this — it's the
difference between a scorer that works and a scorer that works for the rules we
remembered to implement.

DST rules showing as `unmapped` is expected and intentional: team defense needs
pbp aggregation, and it's out of scope for the projection model.

In [ ]:
cov = scoring_coverage_report(reg, scoring)
print(cov['status'].value_counts().to_string(), "\n")
cov[cov.status != 'unmapped']

## 5. Score every player-week

In [ ]:
reg['custom_points'] = compute_custom_score(reg, scoring)

reg[reg.position.isin(SKILL_POSITIONS)][
    ['custom_points', 'fantasy_points', 'fantasy_points_ppr']
].describe().round(2)

## 6. Validate against Sleeper's actual results

This is ground truth, not an approximation — Sleeper ran the league, so its
per-player points for a completed week are definitive. Every row where `diff`
is 0 confirms a rule; every non-zero row points at exactly one rule that's
wrong or missing.

In [ ]:
for wk in VALIDATION_WEEKS:
    r = validate_against_sleeper(reg, crosswalk, scoring,
                                 LEAGUE_ID_2025, 2025, wk)
    s = r[r.position.isin(SKILL_POSITIONS)]
    print(f"Wk {wk:>2}: skill {(s['diff'].abs() <= .01).mean():>6.1%} ({len(s):>3})"
          f"  |  all {(r['diff'].abs() <= .01).mean():>6.1%} ({len(r):>3})")

### Any remaining mismatches

Should be empty. If not, each row names a specific rule to investigate — that's
how every rule in the table at the top was found.

In [ ]:
bad = []
for wk in VALIDATION_WEEKS:
    r = validate_against_sleeper(reg, crosswalk, scoring,
                                 LEAGUE_ID_2025, 2025, wk)
    b = r[r['diff'].abs() > 0.01].copy()
    b['week'] = wk
    bad.append(b)

bad = pd.concat(bad, ignore_index=True)
print(f"{len(bad)} mismatched player-weeks")
bad[['week', 'player_display_name', 'position',
     'sleeper_points', 'custom_points', 'diff']]

## 7. Save the scored table

Phase 2b (usage and efficiency features) picks up from here.

In [ ]:
out = PROJECT_ROOT / 'data' / 'processed' / 'weekly_scored.parquet'
reg.to_parquet(out, index=False)
print(f"Wrote {len(reg):,} rows -> {out.relative_to(PROJECT_ROOT)}")

## What's next — Phase 2b

Usage and efficiency features aggregated from play-by-play:

- **Target share, air-yards share** — how much of the offense flows through a player
- **Snap share** — needs the `pfr_player_id` → `gsis_id` crosswalk hop, which
  hasn't been exercised yet. Most likely place for the next surprise.
- **Red-zone touches** — where the touchdowns actually come from
- **aDOT, YAC** — separating volume from efficiency

**Out of scope, deliberately:** K and DST projections. Kicker output depends on
how often the offense stalls in FG range, which is close to noise week to week;
DST would need a team-defense model layered on an offense model. The dashboard
keeps showing Sleeper's numbers for both, labeled as Sleeper's.
